# V06 - Contrato de leitura DMS

**Objetivo:** criar um contrato minimo (container + view), gravar um node tipado, le-lo estritamente pelo `ViewId` (o contrato) e excluir.

**Ideia central:** consumidores leem pela `view`, nunca pela tabela fisica. A view e o contrato estavel.

In [4]:
# Carrega a configuracao local e cria o cliente CDF. / Load local configuration and create the CDF client.
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from cognite.client import ClientConfig, CogniteClient
from cognite.client.credentials import OAuthClientCredentials, Token

env_file = Path('.env')
pkg_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env.example').exists()),
    Path.cwd(),
)

if env_file.exists():
    load_dotenv(env_file)
else:
    for name in ('.env', '.env.example'):
        candidate = pkg_root / name
        if candidate.exists():
            load_dotenv(candidate)
            break

if not os.environ.get('CDF_CLUSTER'):
    os.environ['CDF_CLUSTER'] = input('CDF_CLUSTER: ').strip()
if not os.environ.get('CDF_PROJECT'):
    os.environ['CDF_PROJECT'] = input('CDF_PROJECT: ').strip()

base_url = os.environ.get('CDF_URL', '').rstrip('/') or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
oauth_ready = env_file.exists() and all(
    os.environ.get(key) for key in ('IDP_TOKEN_URL', 'IDP_CLIENT_ID', 'IDP_CLIENT_SECRET')
)
if oauth_ready:
    scopes = os.environ.get('IDP_SCOPES', '').replace(',', ' ').split() or [f'{base_url}/.default']
    audience = os.environ.get('IDP_AUDIENCE', '')
    credentials = OAuthClientCredentials(
        token_url=os.environ['IDP_TOKEN_URL'],
        client_id=os.environ['IDP_CLIENT_ID'],
        client_secret=os.environ['IDP_CLIENT_SECRET'],
        scopes=scopes,
        **({'audience': audience} if audience else {}),
    )
else:
    # Sem .env nesta pasta: usa Bearer Token.
    bearer_token = (os.environ.get('CDF_BEARER_TOKEN') or getpass('CDF_BEARER_TOKEN: ')).strip()
    if bearer_token.lower().startswith('bearer '):
        bearer_token = bearer_token[7:].strip()
    credentials = Token(bearer_token)

client = CogniteClient(ClientConfig(
    client_name='radix-cdf-onboarding-v06',
    base_url=base_url,
    project=os.environ['CDF_PROJECT'],
    credentials=credentials,
))

## 1. Criar contrato minimo e um node de teste

In [13]:
from uuid import uuid4
from cognite.client.data_classes.data_modeling import (
    ContainerApply, ContainerId, MappedPropertyApply, NodeApply, NodeId,
    NodeOrEdgeData, SpaceApply, Text, ViewApply, ViewId,
)

run = uuid4().hex[:8]
sp = f'sp_ur_training_v06_{run}'
container_id = ContainerId(sp, 'Sensor')
view_id = ViewId(sp, 'Sensor', 'v1')
node_id = NodeId(sp, 'sensor-001')

client.data_modeling.spaces.apply(SpaceApply(space=sp, name='UR training - V06'))
client.data_modeling.containers.apply(ContainerApply._load({
    'space': sp,
    'externalId': container_id.external_id,
    'properties': {'name': {'type': Text().dump(), 'nullable': False}},
    'usedFor': 'node',
}))
client.data_modeling.views.apply(ViewApply(
    space=sp, external_id=view_id.external_id, version=view_id.version,
    properties={'name': MappedPropertyApply(container=container_id, container_property_identifier='name')},
))
client.data_modeling.instances.apply(nodes=NodeApply(
    space=sp, external_id=node_id.external_id,
    sources=[NodeOrEdgeData(source=view_id, properties={'name': 'Sensor de vibracao'})],
))
print('contrato e node criados em', sp)

contrato e node criados em sp_ur_training_v06_f80359e7


## 2. Ler pelo contrato (ViewId)
A leitura passa o `sources=view_id`: so os campos do contrato aparecem.

In [14]:
node = client.data_modeling.instances.retrieve_nodes(nodes=node_id, sources=view_id)
assert node is not None, 'Node nao recuperado pelo contrato.'
print('propriedades expostas pelo contrato:', node.properties[view_id])

nodes_via_view = client.data_modeling.instances.list(instance_type='node', sources=view_id, space=sp, limit=10)
nodes_via_view.to_pandas()

propriedades expostas pelo contrato: {'name': 'Sensor de vibracao'}


,space,external_id,version,last_updated_time,created_time,instance_type,properties
0,sp_ur_training_v06_f80359e7,sensor-001,1,2026-06-24 17:48:19.030,2026-06-24 17:48:19.030,node,{'sp_ur_training_v06_f80359e7': {'Sensor/v1': ...


## Mini-exercicio
- Adicione `unit` (Text) ao container e a view, regrave o node com `unit='mm/s'` e leia de novo pelo contrato.
- Tente ler o node SEM `sources=view_id` e compare o que volta.

## 3. Limpeza idempotente

In [15]:
assert sp.startswith('sp_ur_training_v06_')
client.data_modeling.instances.delete(nodes=node_id)
print('node_still_exists:', client.data_modeling.instances.retrieve(node_id) is not None)
client.data_modeling.views.delete(view_id)
print('view_still_exists:', client.data_modeling.views.retrieve(view_id) is not None)
client.data_modeling.containers.delete(container_id)
print('container_still_exists:', client.data_modeling.containers.retrieve(container_id) is not None)
client.data_modeling.spaces.delete(sp)
print('space_still_exists:', client.data_modeling.spaces.retrieve(sp) is not None)

node_still_exists: True
view_still_exists: True
container_still_exists: False
space_still_exists: False
